In [1]:
from openai import OpenAI

openai_client = OpenAI()

## Question 1. Define the Search Tool

In [2]:
import requests

def search_wikipedia(query):
    """Search Wikipedia and return a list of matching article titles/snippets."""
    
    url = f"https://en.wikipedia.org/w/api.php?action=query&format=json&list=search&srsearch={query}"
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    data = response.json()

    return data

In [3]:
results = search_wikipedia("capybara")

In [4]:
results["query"]["search"]

[{'ns': 0,
  'title': 'Capybara',
  'pageid': 6776,
  'size': 37074,
  'wordcount': 3734,
  'snippet': 'The <span class="searchmatch">capybara</span> or greater <span class="searchmatch">capybara</span> (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South America except Chile. It is',
  'timestamp': '2026-04-17T08:54:32Z'},
 {'ns': 0,
  'title': 'Capybara (disambiguation)',
  'pageid': 69085306,
  'size': 516,
  'wordcount': 100,
  'snippet': 'Look up <span class="searchmatch">capybara</span>\xa0or <span class="searchmatch">Capybara</span> in Wiktionary, the free dictionary. The <span class="searchmatch">capybara</span> is a giant cavy rodent native to South America. <span class="searchmatch">Capybara</span> may also refer to:',
  'timestamp': '2025-04-21T23:08:14Z'},
 {'ns': 0,
  'title': 'Lesser capybara',
  'pageid': 23188846,
  'size': 5578,
  'wordcount': 581,
  'snippet': 'The lesser <span class="searchmatch">capybara</span> (Hydrochoerus ist

**How many total results did the search return? 10**

## Question 2. Analyzing Search Results

In [5]:
count = 0

for result in results["query"]["search"]:
    title = result["title"]
    if "capybara" in title.lower():
        count += 1

print(count)

5


**How many of the results contain the word "capybara" (case-insensitive) in their title? 5**

## Question 3. Define the Get Page Tool

In [6]:
def get_page(title):
    """Fetch the raw wikitext of a Wikipedia page by title."""
    
    url = f"https://en.wikipedia.org/w/index.php?title={title}&action=raw"
    headers = {"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"}

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    return response.text

In [7]:
content = get_page("Capybara")

In [8]:
len(content)

36946

**How many characters are in the page content? 35,000**

## Question 4. Agent Setup

In [9]:
!uv add pydantic-ai

Resolved 229 packages in 6ms
Checked 222 packages in 32ms


In [27]:
import requests
from urllib.parse import quote
from pydantic_ai import Agent


HEADERS = {
    "User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"
}


def search_wikipedia(query: str) -> dict:
    """
    Search Wikipedia for pages matching the query.
    """
    encoded_query = quote(query)
    url = (
        "https://en.wikipedia.org/w/api.php"
        f"?action=query&format=json&list=search&srsearch={encoded_query}"
    )

    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()

    return response.json()


def get_page(title: str) -> str:
    """
    Get the raw wiki text for a Wikipedia page by title.
    """
    encoded_title = quote(title.replace(" ", "_"))

    url = f"https://en.wikipedia.org/w/index.php?title={encoded_title}&action=raw"

    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()

    return response.text


wiki_agent = Agent(
    "openai:gpt-4.1-mini",
    instructions="""
    You are a Wikipedia research agent.

    Use search_wikipedia to find relevant Wikipedia pages.
    Use get_page to read the raw content of a selected page.
    Base your answer on the Wikipedia content you retrieve.
    If search results are unclear, say so.
    """,
    tools=[search_wikipedia, get_page],
)

In [26]:
result = await wiki_agent.run("What is the main topic of the Capybara page")

print(result.output)

The main topic of the Capybara page is the capybara (Hydrochoerus hydrochaeris), which is the largest living rodent native to all countries in South America except Chile. The page provides detailed information about the capybara's classification, physical description, ecology, diet, social behavior, reproduction, communication, conservation status, human interaction, and its presence in popular culture. It highlights the capybara as a semi-aquatic herbivore that lives near water, is highly social, and is hunted for its meat and hide in some areas. The page also discusses its adaptability to urbanization, presence in zoos, and cultural significance, including its popularity in internet memes.


In [33]:
result2 = await wiki_agent.run("What are the main threats to capybara populations?")

print(result2.output)
messages = result2.new_messages()

The main threats to capybara populations include:

1. Hunting: Capybaras are hunted for their meat and hide as well as grease from their thick fatty skin in some areas. Hunting has reduced their numbers in some regions.

2. Human conflict: Capybaras are sometimes killed by humans who see their grazing as competition for livestock.

Despite these threats, capybaras are not considered a threatened species overall and their population is stable throughout most of their South American range. They are able to breed rapidly and have adapted well to urbanization. Conservation efforts include farming capybaras, which can help protect wetland habitats where they live. 

Sources:
- Capybara Wikipedia page, section "Conservation and human interaction"


In [34]:
tool_call_count = 0

for message in messages:
    for part in message.parts:
        if part.part_kind == "tool-call":
            if part.tool_name in {"search_wikipedia", "get_page"}:
                tool_call_count += 1

print(tool_call_count)

2
